# Exercise 10 — Object detection

## What you are about to see

[Exercise 09](q09_image_classification.ipynb) answered one question per image: **which
digit is this?** The image held exactly one digit, centred, and the answer was one label.

Detection asks more: **which objects are in this image, and where is each one?** An image
can hold several digits, anywhere, or none of the ten classes in a given place. So the
answer is no longer one label but a **list**, with one entry per object, and each entry
says *what* (a class) and *where* (a box).

MNIST has no such images, so the first part of this notebook **builds a detection dataset
out of MNIST digits**: it pastes a few digits at random places on a larger, empty canvas
and records, for each one, its class and the box around it. Because we place the digits
ourselves, the answers are exact — no human had to draw a box.

The one thing to take from this part is the **contract of the dataset**: exactly what a
detector receives (the *input*) and exactly what it must produce (the *output*).

In [ ]:
# Run me first: make ``dlcb`` and ``datasets`` importable whether Jupyter was
# started in the project root or inside ``exercises/``.
import pathlib
import sys

ROOT = pathlib.Path.cwd()
if not (ROOT / "dlcb").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from typing import List, Tuple

import matplotlib.patches as patches
import matplotlib.pyplot as plt
import numpy as np

import time

import datasets
from dlcb import engine as cpu
from dlcb import nn, optim
from dlcb.conv import Conv2d, MaxPool2d
from dlcb.loss import cross_entropy

print("ready — project imported from", ROOT)

## 1. Input and output, before any code

**Input — one image.** A grey canvas of $64 \times 64$ pixels, values in $[0, 1]$ (0 is
background, 1 is ink), channels-last like Exercise 09:

$$\mathbf{X} \in [0, 1]^{H \times W \times 1}, \qquad H = W = 64 .$$

**Output — a list of objects**, between 1 and `MAX_OBJECTS` of them, each a pair

$$\bigl(\, c,\; (x_{\min},\, y_{\min},\, x_{\max},\, y_{\max}) \,\bigr), \qquad c \in \{0, \dots, 9\} .$$

The **box** is written in pixels, with one convention used everywhere in this notebook:

- the origin is the **top-left** corner of the image;
- $x$ is the **column** (grows to the right), $y$ is the **row** (grows downwards) — the
  usual image convention, and the opposite order of the array index `image[row, column]`;
- the box is **half-open**: it covers columns $x_{\min} \le x < x_{\max}$ and rows
  $y_{\min} \le y < y_{\max}$. So its width is $x_{\max} - x_{\min}$, its height is
  $y_{\max} - y_{\min}$, and the NumPy slice `image[y_min:y_max, x_min:x_max]` is exactly
  the pixels inside it.

```
 (0,0) ────────────── x (column) ──────────────▶ 64
   │
   │        (x_min, y_min)
   │              ┌──────────┐
   y              │  digit   │  height = y_max − y_min
 (row)            │    7     │
   │              └──────────┘
   │                     (x_max, y_max)   ← first column / row *outside* the box
   ▼                 width = x_max − x_min
  64
```

The box is **tight**: it hugs the ink of the digit, not the $28 \times 28$ MNIST frame the
digit came in. A box is a claim about where the *object* is, and the frame's empty margin is
not part of the object.

The length of the list changes from image to image — that is what makes detection harder
to batch than classification. Section 4 shows how the dataset stores it anyway, in
fixed-size arrays.

In [ ]:
CANVAS = 64             # the image is CANVAS × CANVAS pixels
MIN_OBJECTS = 1         # digits per image: between MIN_OBJECTS and MAX_OBJECTS
MAX_OBJECTS = 3
GAP = 2                 # minimum empty pixels between two boxes (no overlap, no touching)
N_TRAIN = 5_000         # detection images to build from MNIST's training digits
N_TEST = 1_000          # ... and from MNIST's test digits
SEED = 0

## 2. Building one image, step by step

Three small functions, each doing one thing:

1. `tight_crop` cuts the empty margin off an MNIST digit, leaving only the rows and columns
   that contain ink. Its shape becomes the size of the box.
2. `place_digits` picks where each digit goes: a random top-left corner such that the digit
   fits inside the canvas and its box keeps at least `GAP` pixels from every box already
   placed. Non-overlapping objects keep the answers unambiguous: every ink pixel belongs to
   exactly one object.
3. `make_image` pastes the crops onto an empty canvas and returns the input and its output.

In [ ]:
def tight_crop(digit: np.ndarray) -> np.ndarray:
    """Cut a (28, 28) MNIST digit down to the rows and columns that contain ink."""
    rows = np.flatnonzero(digit.max(axis=1) > 0)
    cols = np.flatnonzero(digit.max(axis=0) > 0)
    return digit[rows[0]:rows[-1] + 1, cols[0]:cols[-1] + 1]


def boxes_too_close(a: np.ndarray, b: np.ndarray, gap: int) -> bool:
    """True if boxes a and b, each (x_min, y_min, x_max, y_max), are closer than ``gap``."""
    return not (a[2] + gap <= b[0] or b[2] + gap <= a[0] or
                a[3] + gap <= b[1] or b[3] + gap <= a[1])


def place_digits(crops: List[np.ndarray], rng: np.random.RandomState,
                 max_tries: int = 100) -> List[np.ndarray]:
    """Choose a box for each crop: inside the canvas, at least GAP pixels from the others.

    Returns one box (x_min, y_min, x_max, y_max) per crop, or raises if a crop does not
    fit after ``max_tries`` random positions (the caller then draws new digits).
    """
    boxes: List[np.ndarray] = []
    for crop in crops:
        h, w = crop.shape
        for _ in range(max_tries):
            x0 = rng.randint(0, CANVAS - w + 1)
            y0 = rng.randint(0, CANVAS - h + 1)
            box = np.array([x0, y0, x0 + w, y0 + h])
            if not any(boxes_too_close(box, other, GAP) for other in boxes):
                boxes.append(box)
                break
        else:
            raise RuntimeError("no room left on the canvas")
    return boxes


def make_image(digits: np.ndarray, labels: np.ndarray,
               rng: np.random.RandomState) -> Tuple[np.ndarray, List[Tuple[int, np.ndarray]]]:
    """Paste a random number of random digits onto an empty canvas.

    ``digits`` (n, 28, 28) and ``labels`` (n,) are the MNIST pool to draw from.
    Returns the INPUT, an image (CANVAS, CANVAS, 1), and the OUTPUT, a list of
    (class, box) pairs — one per digit pasted.
    """
    while True:
        k = rng.randint(MIN_OBJECTS, MAX_OBJECTS + 1)
        picks = rng.choice(len(digits), size=k, replace=False)
        crops = [tight_crop(digits[i]) for i in picks]
        try:
            boxes = place_digits(crops, rng)
            break
        except RuntimeError:
            continue                                    # rare: draw other digits
    image = np.zeros((CANVAS, CANVAS, 1))
    for crop, (x0, y0, x1, y1) in zip(crops, boxes):
        image[y0:y1, x0:x1, 0] = crop                   # rows are y, columns are x
    objects = [(int(labels[i]), box) for i, box in zip(picks, boxes)]
    return image, objects

Build one image — we keep drawing until one has three digits, to have something to look
at — and print its output exactly as a detector would have to produce it:

In [ ]:
mnist_train = datasets.load_mnist("train", limit=20_000)
rng = np.random.RandomState(SEED)
while True:
    image, objects = make_image(mnist_train.images[..., 0], mnist_train.labels, rng)
    if len(objects) == MAX_OBJECTS:
        break

print(f"INPUT   image: shape {image.shape}, dtype {image.dtype}, "
      f"values in [{image.min():.0f}, {image.max():.0f}]")
print(f"OUTPUT  {len(objects)} object(s):")
for c, (x0, y0, x1, y1) in objects:
    print(f"        class {c}   box (x_min={x0:2d}, y_min={y0:2d}, x_max={x1:2d}, y_max={y1:2d})"
          f"   -> {x1 - x0} × {y1 - y0} pixels")

And draw it: on the left the **input** — all a detector will ever see; on the right the
same image with the **output** drawn over it, one box and one class per object.

In [ ]:
def draw(ax, image: np.ndarray, objects=(), color: str = "tab:red", title: str = "") -> None:
    """Show an image and, if given, its (class, box) pairs as rectangles with labels."""
    ax.imshow(image[..., 0], cmap="gray_r", vmin=0, vmax=1)
    for c, (x0, y0, x1, y1) in objects:
        # matplotlib's pixel (column x, row y) has its centre at (x, y), so a pixel's
        # edges are at ±0.5: the half-open box [x0, x1) spans x0 − 0.5 … x1 − 0.5.
        ax.add_patch(patches.Rectangle((x0 - 0.5, y0 - 0.5), x1 - x0, y1 - y0,
                                       fill=False, edgecolor=color, linewidth=1.5))
        ax.text(x0 - 0.5, y0 - 1.5, str(c), color=color, fontsize=10, fontweight="bold",
                bbox=dict(facecolor="white", edgecolor="none", pad=0.3))
    ax.set_title(title, fontsize=10)
    ax.set_xticks([0, 16, 32, 48, 63])
    ax.set_yticks([0, 16, 32, 48, 63])


fig, (left, right) = plt.subplots(1, 2, figsize=(8, 4))
draw(left, image, title="INPUT: the image")
draw(right, image, objects, title="OUTPUT: class + box per digit")
left.set_xlabel("x (column)"); left.set_ylabel("y (row)")
right.set_xlabel("x (column)")
plt.tight_layout()
plt.show()

## 3. The whole dataset

Training images are built **only** from MNIST's training digits and test images **only**
from MNIST's test digits — written by different people. A digit seen in training never
reappears at test time, inside any image, so the test measures detection of new
handwriting, not memory of familiar strokes.

In [ ]:
mnist_test = datasets.load_mnist("test")


def build_split(mnist, n_images: int, seed: int):
    """``n_images`` detection images from one MNIST split: a list of (image, objects)."""
    rng = np.random.RandomState(seed)
    digits, labels = mnist.images[..., 0], mnist.labels
    return [make_image(digits, labels, rng) for _ in range(n_images)]


train_pairs = build_split(mnist_train, N_TRAIN, SEED)
test_pairs = build_split(mnist_test, N_TEST, SEED + 1)
print(f"train: {len(train_pairs)} images from MNIST train digits")
print(f"test:  {len(test_pairs)} images from MNIST test digits")

## 4. Storing a variable-length output in fixed-size arrays

A list of lists is clear but cannot be stacked into one array for a batch. The standard
remedy is **padding**: give every image `MAX_OBJECTS` slots, fill the first ones with its
objects, and mark the rest as empty. The dataset is then four arrays, all indexed by the
image number `i`:

| array | shape | dtype | meaning |
|---|---|---|---|
| `images` | `(N, 64, 64, 1)` | float | **input**: the canvas, values in $[0, 1]$ |
| `boxes` | `(N, MAX_OBJECTS, 4)` | int | **output**: `boxes[i, j]` = $(x_{\min}, y_{\min}, x_{\max}, y_{\max})$ of object $j$ |
| `labels` | `(N, MAX_OBJECTS)` | int | **output**: `labels[i, j]` = class of object $j$, or **−1** for an empty slot |
| `counts` | `(N,)` | int | number of real objects in image $i$: the slots `0 … counts[i] − 1` |

Empty slots have label −1 and box `(0, 0, 0, 0)`; **only `labels` decides whether a slot is
real** — never read a box without checking its label. Objects fill the slots in the order
they were pasted, which means nothing: a detector's output is a *set*, and any order of the
same objects is the same answer.

In [ ]:
def to_arrays(pairs) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """List of (image, objects) -> images, boxes, labels, counts (padded with -1)."""
    n = len(pairs)
    images = np.stack([img for img, _ in pairs])                    # (N, 64, 64, 1)
    boxes = np.zeros((n, MAX_OBJECTS, 4), dtype=np.int64)
    labels = np.full((n, MAX_OBJECTS), -1, dtype=np.int64)
    counts = np.zeros(n, dtype=np.int64)
    for i, (_, objects) in enumerate(pairs):
        counts[i] = len(objects)
        for j, (c, box) in enumerate(objects):
            labels[i, j], boxes[i, j] = c, box
    return images, boxes, labels, counts


def objects_of(i: int, boxes: np.ndarray, labels: np.ndarray) -> List[Tuple[int, np.ndarray]]:
    """Back from the arrays to the list form: the real (class, box) pairs of image i."""
    return [(int(labels[i, j]), boxes[i, j]) for j in range(MAX_OBJECTS) if labels[i, j] >= 0]


train_images, train_boxes, train_labels, train_counts = to_arrays(train_pairs)
test_images, test_boxes, test_labels, test_counts = to_arrays(test_pairs)

for name, arr in (("train_images", train_images), ("train_boxes", train_boxes),
                  ("train_labels", train_labels), ("train_counts", train_counts)):
    print(f"{name:13} shape {str(arr.shape):18} dtype {arr.dtype}")

i = int(np.flatnonzero(train_counts == 2)[0])        # an image with two digits: one empty slot
print(f"\nimage {i}:  counts = {train_counts[i]}")
print("labels =", train_labels[i].tolist())
print("boxes  =", train_boxes[i].tolist())
print(f"objects_of({i}) =", [(c, b.tolist()) for c, b in objects_of(i, train_boxes, train_labels)])

## 5. Checking the contract

Everything a detector will be trained and judged on comes from these arrays, so it is worth
checking, in code, that they mean what Sections 1 and 4 say they mean. Each check below is
one sentence of the contract:

In [ ]:
def check_dataset(images, boxes, labels, counts) -> None:
    n = len(images)
    real = labels >= 0                                            # (N, MAX_OBJECTS)

    # Shapes and value ranges.
    assert images.shape == (n, CANVAS, CANVAS, 1) and images.min() >= 0 and images.max() <= 1
    # Every image has MIN_OBJECTS..MAX_OBJECTS objects, in the first `counts` slots.
    assert np.all((counts >= MIN_OBJECTS) & (counts <= MAX_OBJECTS))
    assert np.array_equal(real.sum(axis=1), counts)
    assert np.all(real == (np.arange(MAX_OBJECTS) < counts[:, None]))
    # Real labels are digits; empty slots are -1 with a zero box.
    assert np.all((labels[real] >= 0) & (labels[real] <= 9))
    assert np.all(boxes[~real] == 0)
    # Real boxes are non-empty and inside the canvas (half-open: x_max may equal CANVAS).
    x0, y0, x1, y1 = (boxes[real][:, k] for k in range(4))
    assert np.all((0 <= x0) & (x0 < x1) & (x1 <= CANVAS))
    assert np.all((0 <= y0) & (y0 < y1) & (y1 <= CANVAS))

    for i in range(n):
        objects = objects_of(i, boxes, labels)
        ink = images[i, :, :, 0] > 0
        covered = np.zeros_like(ink)
        for c, (bx0, by0, bx1, by1) in objects:
            inside = ink[by0:by1, bx0:bx1]
            # Tight: the box's first and last row and column each contain ink.
            assert inside[0].any() and inside[-1].any() and inside[:, 0].any() and inside[:, -1].any()
            covered[by0:by1, bx0:bx1] = True
        # No ink outside the boxes: every inked pixel belongs to some object.
        assert not np.any(ink & ~covered)
        # No two boxes closer than GAP pixels.
        for a in range(len(objects)):
            for b in range(a + 1, len(objects)):
                assert not boxes_too_close(objects[a][1], objects[b][1], GAP)


for name, split in (("train", (train_images, train_boxes, train_labels, train_counts)),
                    ("test", (test_images, test_boxes, test_labels, test_counts))):
    check_dataset(*split)
    print(f"{name}: every check passed")

### A look at many images, and at the numbers

A grid of training images with their boxes, and three statistics a detector designer would
want to know first: how many objects per image, how big the boxes are, and whether every
class is represented.

In [ ]:
fig, axes = plt.subplots(3, 6, figsize=(12, 6.4))
for k, ax in enumerate(axes.flat):
    draw(ax, train_images[k], objects_of(k, train_boxes, train_labels), title=f"image {k}")
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout()
plt.show()

real = train_labels >= 0
widths = (train_boxes[..., 2] - train_boxes[..., 0])[real]
heights = (train_boxes[..., 3] - train_boxes[..., 1])[real]
print("objects per image:", {k: int((train_counts == k).sum()) for k in range(MIN_OBJECTS, MAX_OBJECTS + 1)})
print(f"box width:  {widths.min()}–{widths.max()} px (mean {widths.mean():.1f})")
print(f"box height: {heights.min()}–{heights.max()} px (mean {heights.mean():.1f})")
print("objects per class:", np.bincount(train_labels[real], minlength=10).tolist())
print(f"area covered by boxes: {(widths * heights).sum() / (len(train_images) * CANVAS ** 2):.1%} of all pixels")

---

**The contract, in one table.** This is all a detector gets and all it is asked for:

| | what | shape |
|---|---|---|
| **input** | one image, grey levels in $[0, 1]$ | `(64, 64, 1)` |
| **output** | a set of 1–3 objects, each a class $c \in \{0,\dots,9\}$ and a box $(x_{\min}, y_{\min}, x_{\max}, y_{\max})$ in pixels, half-open, origin top-left | stored as `labels (3,)` with −1 = empty, `boxes (3, 4)` |

The test images, built from digits the training images never used, are where any detector
trained on this data will be judged.

# Part 2 — A detector

The contract says: one image in, a *set* of (class, box) pairs out, between one and three
of them. A network, however, always outputs an array of the **same shape**. The whole
design of a detector is the answer to one question: **how do we write a variable-length
set of objects as a fixed-size array** — so that a network can produce it and a loss can
compare it with the truth?

## 6. The output: a grid of cells, 15 numbers per cell

The idea used here is the one of YOLO (Redmon et al., 2016). Cut the $64 \times 64$ image
into a $G \times G = 8 \times 8$ **grid** of cells, each $8 \times 8$ pixels. The network
outputs **one vector of 15 numbers per cell** — an array of shape `(8, 8, 15)` per image,
whatever is in it.

An object is assigned to **one cell: the cell that contains the centre of its box**. That
cell is *responsible* for the object; it must say "there is an object here", where its box
is, and which digit it is. Every other cell must say "nothing is centred here". In this
dataset two digits never share a cell (the next cell checks it), so one object per cell is
enough.

The 15 numbers of a cell, in order:

| channels | name | what the cell is asked | turned into a value by | target in a responsible cell | target elsewhere |
|---|---|---|---|---|---|
| 0 | $z_{\mathrm{obj}}$ | is the centre of an object in me? | $p_{\mathrm{obj}} = \sigma(z_{\mathrm{obj}})$ | 1 | **0** |
| 1, 2 | $t_x, t_y$ | where in me is the centre? | $\sigma(t_x), \sigma(t_y) \in (0, 1)$: fraction of the cell | $c_x / 8 - \text{col}$, $\;c_y / 8 - \text{row}$ | — (ignored) |
| 3, 4 | $t_w, t_h$ | how big is the box? | $\sigma(t_w), \sigma(t_h) \in (0, 1)$: fraction of the image | $w / 64$, $\;h / 64$ | — (ignored) |
| 5 … 14 | $s_0, \dots, s_9$ | which digit is it? | $\operatorname{softmax}$: probability of each class | the class $c$ | — (ignored) |

Here $(c_x, c_y)$ is the centre of the box in pixels, $w$ and $h$ its width and height:
$c_x = (x_{\min} + x_{\max}) / 2$, $w = x_{\max} - x_{\min}$, and the same for $y$.

**Decoding** — going from the 15 numbers of the cell in row $r$, column $k$ back to a box
in pixels — inverts those definitions:

$$
c_x = 8\,\bigl(k + \sigma(t_x)\bigr), \qquad c_y = 8\,\bigl(r + \sigma(t_y)\bigr), \qquad
w = 64\,\sigma(t_w), \qquad h = 64\,\sigma(t_h),
$$
$$
(x_{\min}, y_{\min}, x_{\max}, y_{\max}) = \bigl(c_x - \tfrac{w}{2},\; c_y - \tfrac{h}{2},\;
c_x + \tfrac{w}{2},\; c_y + \tfrac{h}{2}\bigr).
$$

Two things to notice. Each number is squashed into the range where its answer lives: a
centre offset *must* lie inside the cell, so it goes through a sigmoid. And channels 1–14
mean something **only in a responsible cell**: in the other cells nobody will ever read
them, so the loss will not train them there.

In [ ]:
GRID = 8                                # the output is a GRID × GRID map of cells
CELL = CANVAS // GRID                   # each cell covers CELL × CELL pixels
N_CLASSES = 10
OBJ, BOX, CLS = 0, slice(1, 5), slice(5, 5 + N_CLASSES)    # where each part sits in a cell
N_OUT = 1 + 4 + N_CLASSES                                   # 15 numbers per cell
IGNORE = -100                           # class target of a cell with no object


def encode(boxes: np.ndarray, labels: np.ndarray):
    """Dataset arrays -> what each cell should output.

    Returns three arrays over the grid:
      obj  (N, GRID, GRID)     1 in a responsible cell, 0 elsewhere      (target of channel 0)
      box  (N, GRID, GRID, 4)  (offset x, offset y, w/64, h/64)          (targets of channels 1-4)
      cls  (N, GRID, GRID)     the class, or IGNORE where obj == 0       (target of channels 5-14)
    """
    n = len(labels)
    obj = np.zeros((n, GRID, GRID))
    box = np.zeros((n, GRID, GRID, 4))
    cls = np.full((n, GRID, GRID), IGNORE)
    for i in range(n):
        for c, (x0, y0, x1, y1) in objects_of(i, boxes, labels):
            cx, cy = (x0 + x1) / 2, (y0 + y1) / 2              # centre, in pixels
            col, row = int(cx // CELL), int(cy // CELL)        # the responsible cell
            assert obj[i, row, col] == 0, "two objects in one cell"
            obj[i, row, col] = 1
            box[i, row, col] = [cx / CELL - col, cy / CELL - row,
                                (x1 - x0) / CANVAS, (y1 - y0) / CANVAS]
            cls[i, row, col] = c
    return obj, box, cls


def cell_to_box(row: int, col: int, t: np.ndarray) -> np.ndarray:
    """Four values in (0, 1) of cell (row, col) -> box (x_min, y_min, x_max, y_max) in pixels."""
    cx, cy = CELL * (col + t[0]), CELL * (row + t[1])
    w, h = CANVAS * t[2], CANVAS * t[3]
    return np.array([cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2])


train_obj, train_box, train_cls = encode(train_boxes, train_labels)
test_obj, test_box, test_cls = encode(test_boxes, test_labels)
print(f"targets: obj {train_obj.shape}, box {train_box.shape}, cls {train_cls.shape}")
print(f"responsible cells: {int(train_obj.sum())} of {train_obj.size} "
      f"({train_obj.mean():.1%}); no cell holds two objects")

### One image, cell by cell

Take a training image with three digits. The figure draws the $8 \times 8$ grid over it,
marks each box centre with a dot, and shades the cell that contains it — the responsible
cell. Below, the target vector of every responsible cell, number by number, and a check
that decoding it gives back exactly the box we started from.

In [ ]:
i = int(np.flatnonzero(train_counts == 3)[0])
objects = objects_of(i, train_boxes, train_labels)

fig, ax = plt.subplots(figsize=(5, 5))
draw(ax, train_images[i], objects, title=f"training image {i}: grid of {GRID}×{GRID} cells")
for k in range(1, GRID):
    ax.axhline(k * CELL - 0.5, color="tab:blue", linewidth=0.5)
    ax.axvline(k * CELL - 0.5, color="tab:blue", linewidth=0.5)
for row, col in zip(*np.nonzero(train_obj[i])):
    ax.add_patch(patches.Rectangle((col * CELL - 0.5, row * CELL - 0.5), CELL, CELL,
                                   color="tab:blue", alpha=0.25))
for c, (x0, y0, x1, y1) in objects:
    ax.plot((x0 + x1) / 2 - 0.5, (y0 + y1) / 2 - 0.5, "o", color="tab:blue")
ax.set_xticks(np.arange(0, CANVAS + 1, CELL) - 0.5, labels=[str(v) for v in range(0, CANVAS + 1, CELL)])
ax.set_yticks(np.arange(0, CANVAS + 1, CELL) - 0.5, labels=[str(v) for v in range(0, CANVAS + 1, CELL)])
plt.tight_layout()
plt.show()

for c, (x0, y0, x1, y1) in objects:
    cx, cy = (x0 + x1) / 2, (y0 + y1) / 2
    col, row = int(cx // CELL), int(cy // CELL)
    t = train_box[i, row, col]
    print(f"digit {c}: box ({x0}, {y0}, {x1}, {y1}), centre ({cx}, {cy}) -> cell row {row}, col {col}")
    print(f"   target  ch 0      objectness = {train_obj[i, row, col]:.0f}")
    print(f"           ch 1, 2   centre in cell = ({t[0]:.4f}, {t[1]:.4f})")
    print(f"           ch 3, 4   size / 64      = ({t[2]:.4f}, {t[3]:.4f})")
    print(f"           ch 5-14   class          = {train_cls[i, row, col]}  (one-hot over the 10 logits)")
    print(f"   decoded back: {cell_to_box(row, col, t)}\n")
print(f"The other {GRID * GRID - len(objects)} cells: objectness target 0, everything else ignored.")

# The round trip is exact for every object of every image, train and test.
for obj, box, cls, boxes, labels in ((train_obj, train_box, train_cls, train_boxes, train_labels),
                                     (test_obj, test_box, test_cls, test_boxes, test_labels)):
    for n in range(len(labels)):
        decoded = sorted(tuple(cell_to_box(r, k, box[n, r, k])) + (cls[n, r, k],)
                         for r, k in zip(*np.nonzero(obj[n])))
        truth = sorted(tuple(map(float, b)) + (c,) for c, b in objects_of(n, boxes, labels))
        assert decoded == truth
print("round trip boxes -> targets -> boxes: exact for all train and test images")

## 7. The network: a convolutional backbone and one $1 \times 1$ head

The network has to turn a $64 \times 64 \times 1$ image into an $8 \times 8 \times 15$ map.
Three rounds of *convolution → ReLU → max pooling* halve the resolution three times,
$64 \to 32 \to 16 \to 8$, one more convolution mixes the features at that resolution, and
the **head** produces the 15 numbers:

| layer | output | parameters |
|---|---|---|
| input | $64 \times 64 \times 1$ | — |
| `Conv2d(1, 8, 3, padding=1)` + ReLU, `MaxPool2d(2)` | $32 \times 32 \times 8$ | $(9 + 1) \cdot 8 = 80$ |
| `Conv2d(8, 16, 3, padding=1)` + ReLU, `MaxPool2d(2)` | $16 \times 16 \times 16$ | $(72 + 1) \cdot 16 = 1\,168$ |
| `Conv2d(16, 32, 3, padding=1)` + ReLU, `MaxPool2d(2)` | $8 \times 8 \times 32$ | $(144 + 1) \cdot 32 = 4\,640$ |
| `Conv2d(32, 32, 3, padding=1)` + ReLU | $8 \times 8 \times 32$ | $(288 + 1) \cdot 32 = 9\,248$ |
| **head** `Conv2d(32, 15, 1)` | $8 \times 8 \times 15$ | $(32 + 1) \cdot 15 = 495$ |

**The head is a $1 \times 1$ convolution**, which is a dense layer applied to every cell
separately (Lecture V, *Convolução 2D, 2D multicanal e 3D*): its weight matrix
$\mathbf{W} \in \mathbb{R}^{33 \times 15}$ turns the 32 features of a cell (plus the bias
row) into that cell's 15 outputs, and **the same $\mathbf{W}$ serves all 64 cells**. So no
cell is special: a digit at the top-left is detected by exactly the same weights as one at
the bottom-right — the weight sharing of Exercise 09, now at the output.

**Each cell sees enough.** A cell's 15 numbers depend on a $38 \times 38$-pixel window of
the image (its *receptive field*: each $3 \times 3$ convolution widens it by two steps of
the current resolution, each pooling doubles the step), and the largest digit is 20 pixels
across — so the responsible cell sees its whole digit, and some of its neighbourhood.

In [ ]:
class DigitDetector(nn.Module):
    """Convolutional backbone (64×64 -> 8×8 features) and a 1×1 head with 15 outputs per cell."""

    def __init__(self) -> None:
        self.conv1 = Conv2d(1, 8, 3, padding=1)          # 64×64×1  -> 64×64×8  -> pool 32×32
        self.conv2 = Conv2d(8, 16, 3, padding=1)         # 32×32×8  -> 32×32×16 -> pool 16×16
        self.conv3 = Conv2d(16, 32, 3, padding=1)        # 16×16×16 -> 16×16×32 -> pool 8×8
        self.conv4 = Conv2d(32, 32, 3, padding=1)        # 8×8×32   -> 8×8×32
        self.pool = MaxPool2d(2)
        self.head = Conv2d(32, N_OUT, 1)                 # 8×8×32   -> 8×8×15

    def forward(self, x: cpu.Tensor) -> cpu.Tensor:
        h = self.pool(self.conv1(x).relu())
        h = self.pool(self.conv2(h).relu())
        h = self.pool(self.conv3(h).relu())
        h = self.conv4(h).relu()
        return self.head(h)                              # (batch, 8, 8, 15): raw numbers


cpu.set_seed(SEED)
model = DigitDetector()
print(f"parameters: {sum(p.data.size for p in model.parameters()):,}")

with cpu.no_grad():
    out = model(cpu.Tensor(train_images[i:i + 1])).data[0]          # (8, 8, 15)
row, col = np.argwhere(train_obj[i])[0]
print(f"\noutput for one image: shape {out.shape}  ->  {GRID}×{GRID} cells × {N_OUT} numbers")
print(f"cell (row {row}, col {col}), the one responsible for a digit, BEFORE training:")
print(f"   ch 0      z_obj = {out[row, col, OBJ]:+.3f}")
print(f"   ch 1-4    t     = {np.round(out[row, col, BOX], 3)}")
print(f"   ch 5-14   s     = {np.round(out[row, col, CLS], 3)}")
print("Random weights give small numbers: p_obj = σ(z_obj) near 0.5 in every cell, a box of"
      "\narbitrary size near the cell centre, class probabilities near 1/10. Training must shape all 15.")

## 8. The loss: three terms, each reading its own channels

The output and the targets have the same layout, so the loss is the sum of three familiar
pieces, each looking at its own channels. Let $\mathcal{P}$ be the set of responsible cells
in a batch (one per object) and $N_{\mathrm{obj}} = |\mathcal{P}|$.

**Objectness — channel 0, every cell.** A yes/no question per cell, so binary
cross-entropy on $p_{\mathrm{obj}} = \sigma(z_{\mathrm{obj}})$ against the target
$o \in \{0, 1\}$:

$$
\mathcal{L}_{\mathrm{obj}} = \frac{1}{N_{\mathrm{obj}}} \sum_{\text{cells}} \lambda_{o}
\Bigl[ -o \log p_{\mathrm{obj}} - (1 - o) \log (1 - p_{\mathrm{obj}}) \Bigr],
\qquad \lambda_o = \begin{cases} 1 & o = 1 \\ \lambda_{\mathrm{noobj}} = 0.1 & o = 0 \end{cases}
$$

Most cells are empty — about 62 of 64 — so without the weight $\lambda_{\mathrm{noobj}}$
the easiest way to lower this loss would be to say "nothing here" everywhere. In code the
bracket is computed from the logit directly, as
$\max(z, 0) - z\,o + \log(1 + e^{-|z|})$, which is the same quantity but never takes the
log of a number that has rounded to 0.

**Box — channels 1–4, responsible cells only.** Squared error between the four squashed
values and their targets:

$$
\mathcal{L}_{\mathrm{box}} = \frac{\lambda_{\mathrm{box}}}{N_{\mathrm{obj}}}
\sum_{\mathcal{P}} \bigl\lVert \sigma(\mathbf{t}) - \mathbf{t}^{\star} \bigr\rVert^2,
\qquad \lambda_{\mathrm{box}} = 5 .
$$

**Class — channels 5–14, responsible cells only.** The softmax cross-entropy of Exercise
09, at the responsible cells; every other cell has target `IGNORE` and contributes nothing:

$$
\mathcal{L}_{\mathrm{cls}} = \frac{1}{N_{\mathrm{obj}}} \sum_{\mathcal{P}}
-\log \operatorname{softmax}(\mathbf{s})_{c} .
$$

$$
\mathcal{L} = \mathcal{L}_{\mathrm{obj}} + \mathcal{L}_{\mathrm{box}} + \mathcal{L}_{\mathrm{cls}} .
$$

Because the box and class terms are multiplied by zero outside $\mathcal{P}$, channels
1–14 of an empty cell receive **zero gradient**: the network is free to put anything there,
and decoding will never read it. The two weights, $\lambda_{\mathrm{noobj}} = 0.1$ and
$\lambda_{\mathrm{box}} = 5$, are the ones YOLO used for the same two reasons.

In [ ]:
LAMBDA_NOOBJ = 0.1
LAMBDA_BOX = 5.0


def sigmoid_t(z: cpu.Tensor) -> cpu.Tensor:
    """σ(z) from engine ops: σ(z) = (tanh(z/2) + 1) / 2."""
    return (z * 0.5).tanh() * 0.5 + 0.5


def bce_with_logits(z: cpu.Tensor, target: np.ndarray) -> cpu.Tensor:
    """Element-wise binary cross-entropy of σ(z) against 0/1 targets, stable for any z."""
    t = cpu.Tensor(target, requires_grad=False)
    abs_z = z.relu() + (-z).relu()
    return z.relu() - z * t + ((-abs_z).exp() + 1.0).log()


def detection_loss(out: cpu.Tensor, obj: np.ndarray, box: np.ndarray, cls: np.ndarray):
    """L = L_obj + L_box + L_cls for a batch; also returns the three parts as floats."""
    n_obj = obj.sum()
    weight = np.where(obj > 0, 1.0, LAMBDA_NOOBJ)
    l_obj = (bce_with_logits(out[..., OBJ], obj)
             * cpu.Tensor(weight, requires_grad=False)).sum() * (1.0 / n_obj)

    responsible = cpu.Tensor(obj, requires_grad=False)              # 1 where a box exists
    sq_err = ((sigmoid_t(out[..., BOX]) - cpu.Tensor(box, requires_grad=False)) ** 2).sum(axis=-1)
    l_box = (sq_err * responsible).sum() * (LAMBDA_BOX / n_obj)

    l_cls = cross_entropy(out[..., CLS], cls, ignore_index=IGNORE)  # mean over responsible cells

    return l_obj + l_box + l_cls, (float(l_obj.data), float(l_box.data), float(l_cls.data))


with cpu.no_grad():
    loss, parts = detection_loss(model(cpu.Tensor(train_images[:32])),
                                 train_obj[:32], train_box[:32], train_cls[:32])
print(f"untrained, one batch: L = {float(loss.data):.3f}   "
      f"(obj {parts[0]:.3f} + box {parts[1]:.3f} + cls {parts[2]:.3f})")
print(f"for reference: a uniform guess over 10 classes has L_cls = ln 10 = {np.log(10):.3f}")

## 9. Training

The usual four lines, mini-batches of `BATCH` images, Adam. Each epoch prints the three
parts of the loss separately, so we can see which question the network learns first. It
takes about **five minutes**: each image is $64 \times 64$, and every convolution runs at
every position of it.

In [ ]:
BATCH = 32
EPOCHS = 6
LR = 2e-3

opt = optim.Adam(model.parameters(), lr=LR)
cpu.reset_flops()
started = time.time()
print(f"{'epoch':>5}  {'L':>6}  {'L_obj':>6}  {'L_box':>6}  {'L_cls':>6}")
for epoch in range(1, EPOCHS + 1):
    order = np.random.permutation(len(train_images))
    parts = []
    for start in range(0, len(order), BATCH):
        rows = order[start:start + BATCH]
        opt.zero_grad()
        loss, p = detection_loss(model(cpu.Tensor(train_images[rows])),
                                 train_obj[rows], train_box[rows], train_cls[rows])
        loss.backward()
        opt.step()
        parts.append(p)
    o, b, c = np.mean(parts, axis=0)
    print(f"{epoch:>5}  {o + b + c:6.3f}  {o:6.3f}  {b:6.3f}  {c:6.3f}   "
          f"[{time.time() - started:4.0f} s, {cpu.flop_count() / 1e9:5.0f} GFLOP]")

## 10. From 64 × 15 numbers back to a list of objects

Training shaped the raw output; turning it into the answer the contract asks for takes
three steps:

1. **Threshold.** Keep the cells with $p_{\mathrm{obj}} = \sigma(z_{\mathrm{obj}}) > \tau$
   ($\tau = 0.5$). Each kept cell is one *detection*, with score $p_{\mathrm{obj}}$.
2. **Decode** each kept cell with the formulas of Section 6: its box from channels 1–4, its
   class as the arg-max of channels 5–14.
3. **Remove duplicates** (*non-maximum suppression*). A digit whose centre lies near a cell
   border can make two neighbouring cells fire. Sort the detections by score and drop any
   that overlaps an already kept one too much.

"Overlaps too much" is measured by the **intersection over union** of two boxes, the
standard measure of how well two boxes agree:

$$
\operatorname{IoU}(A, B) = \frac{|A \cap B|}{|A \cup B|}
= \frac{|A \cap B|}{|A| + |B| - |A \cap B|} \in [0, 1],
$$

1 for identical boxes, 0 for disjoint ones. The same measure will decide, in Section 11,
whether a detection found its object.

In [ ]:
THRESHOLD = 0.5
NMS_IOU = 0.3


def iou(a: np.ndarray, b: np.ndarray) -> float:
    """Intersection over union of two boxes (x_min, y_min, x_max, y_max)."""
    iw = max(0.0, min(a[2], b[2]) - max(a[0], b[0]))
    ih = max(0.0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = iw * ih
    union = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / union


def decode(out: np.ndarray, threshold: float = THRESHOLD) -> List[dict]:
    """One image's raw output (GRID, GRID, 15) -> list of detections, best score first.

    Boxes are clipped to the canvas: a predicted box may stick out of the image, but the
    object cannot.
    """
    p_obj = 1.0 / (1.0 + np.exp(-out[..., OBJ]))
    t = 1.0 / (1.0 + np.exp(-out[..., BOX]))
    detections = []
    for row, col in zip(*np.nonzero(p_obj > threshold)):
        detections.append({"score": float(p_obj[row, col]),
                           "cls": int(out[row, col, CLS].argmax()),
                           "box": np.clip(cell_to_box(row, col, t[row, col]), 0, CANVAS),
                           "cell": (int(row), int(col))})
    detections.sort(key=lambda d: -d["score"])
    kept = []
    for d in detections:                                     # non-maximum suppression
        if all(iou(d["box"], k["box"]) < NMS_IOU for k in kept):
            kept.append(d)
    return kept


def raw_outputs(images: np.ndarray, chunk: int = 250) -> np.ndarray:
    """Raw (N, GRID, GRID, 15) outputs for many images, without building a graph."""
    with cpu.no_grad():
        return np.concatenate([model(cpu.Tensor(images[k:k + chunk])).data
                               for k in range(0, len(images), chunk)])


test_out = raw_outputs(test_images)
print(f"raw test output: {test_out.shape}")

Follow one test image through the three steps. Left: the objectness map, $p_{\mathrm{obj}}$
for each of the 64 cells. Middle: the 15 numbers of the most confident cell, decoded.
Right: the final list of detections (red) over the true boxes (green).

In [ ]:
def draw_result(ax, k: int, detections: List[dict], title: str = "") -> None:
    """Test image k: true boxes in green, detections in red with their class."""
    draw(ax, test_images[k], objects_of(k, test_boxes, test_labels), color="tab:green", title=title)
    for d in detections:
        x0, y0, x1, y1 = d["box"]
        ax.add_patch(patches.Rectangle((x0 - 0.5, y0 - 0.5), x1 - x0, y1 - y0,
                                       fill=False, edgecolor="tab:red", linewidth=1.5,
                                       linestyle="--"))
        ax.text(x1 - 0.5, y1 + 2.5, str(d["cls"]), color="tab:red", fontsize=10,
                fontweight="bold", ha="right",
                bbox=dict(facecolor="white", edgecolor="none", pad=0.3))


k = int(np.flatnonzero(test_counts == 3)[0])
out = test_out[k]
p_obj = 1.0 / (1.0 + np.exp(-out[..., OBJ]))
detections = decode(out)

fig, (a1, a2) = plt.subplots(1, 2, figsize=(9, 4.3))
im = a1.imshow(p_obj, cmap="viridis", vmin=0, vmax=1)
for (r, c), v in np.ndenumerate(p_obj):
    a1.text(c, r, f"{v:.2f}"[1:] if v < 0.995 else "1", ha="center", va="center", fontsize=7,
            color="white" if v < 0.6 else "black")
a1.set_title("p_obj of every cell (8×8)", fontsize=10)
a1.set_xlabel("col"); a1.set_ylabel("row")
draw_result(a2, k, detections, title=f"test image {k}: green = truth, red = detections")
plt.tight_layout()
plt.show()

best = detections[0]
r, c = best["cell"]
s = out[r, c, CLS]
p_cls = np.exp(s - s.max()) / np.exp(s - s.max()).sum()
t = 1.0 / (1.0 + np.exp(-out[r, c, BOX]))
print(f"the most confident cell: row {r}, col {c}")
print(f"   ch 0      z_obj = {out[r, c, OBJ]:+.2f}   ->  p_obj = σ(z_obj) = {best['score']:.3f}")
print(f"   ch 1-4    t     = {np.round(out[r, c, BOX], 2)}   ->  σ(t) = {np.round(t, 3)}")
print(f"             centre = (8·({c} + {t[0]:.3f}), 8·({r} + {t[1]:.3f})) = "
      f"({CELL * (c + t[0]):.1f}, {CELL * (r + t[1]):.1f}) px,   "
      f"size = (64·{t[2]:.3f}, 64·{t[3]:.3f}) = ({CANVAS * t[2]:.1f}, {CANVAS * t[3]:.1f}) px")
print(f"             box   = {np.round(best['box'], 1)}")
print(f"   ch 5-14   s     -> softmax -> class {best['cls']} with probability {p_cls.max():.3f}")
print(f"\nfinal answer: {len(detections)} detection(s)")
for d in detections:
    print(f"   class {d['cls']}  score {d['score']:.3f}  box {np.round(d['box'], 1)}")
print("truth:")
for c_, b_ in objects_of(k, test_boxes, test_labels):
    print(f"   class {c_}               box {b_}")

## 11. Error analysis

A detector can be wrong in more ways than a classifier, and the ways point at different
parts of its output. We sort every **detection** and every **true object** of the test set
into exactly one category each, following Hoiem, Chodpathumwan and Dai (2012). For each
detection, take the true object it overlaps most; then, going through the detections from
the highest score down:

| detection category | rule | which channels are to blame |
|---|---|---|
| **correct** | IoU ≥ 0.5 with an object not yet claimed, right class | — |
| **wrong class** | IoU ≥ 0.5 with an object not yet claimed, wrong class | class, 5–14 |
| **duplicate** | IoU ≥ 0.5 with an object already claimed by a better detection | objectness, 0 (and NMS) |
| **poor box** | 0.1 ≤ IoU < 0.5: it is near an object but the box is off | box, 1–4 |
| **background** | IoU < 0.1 with every object: it fired on nothing | objectness, 0 |

and for each true object:

| object category | rule |
|---|---|
| **found** | claimed by a *correct* detection |
| **found, wrong class** | claimed by a *wrong class* detection |
| **poorly localised** | not claimed, but some detection has 0.1 ≤ IoU < 0.5 with it |
| **missed** | no detection comes near it — channel 0 stayed below the threshold |

**Precision** is the fraction of detections that are correct; **recall** is the fraction of
objects that are found. The threshold 0.5 on IoU is the usual one (PASCAL VOC).

In [ ]:
def analyse(outs: np.ndarray, boxes: np.ndarray, labels: np.ndarray,
            threshold: float = THRESHOLD) -> Tuple[list, list]:
    """Categorise every detection and every true object; return both lists of records."""
    det_records, obj_records = [], []
    for n in range(len(outs)):
        truth = objects_of(n, boxes, labels)
        detections = decode(outs[n], threshold)
        claimed_by = [None] * len(truth)                 # category of the claiming detection
        near = [False] * len(truth)                      # some detection with 0.1 <= IoU < 0.5
        for d in detections:                             # best score first
            ious = [iou(d["box"], b) for _, b in truth]
            j = int(np.argmax(ious))
            best = ious[j]
            if best >= 0.5:
                if claimed_by[j] is None:
                    kind = "correct" if d["cls"] == truth[j][0] else "wrong class"
                    claimed_by[j] = kind
                else:
                    kind = "duplicate"
            elif best >= 0.1:
                kind = "poor box"
                near[j] = True
            else:
                kind = "background"
            det_records.append({"image": n, "kind": kind, "det": d, "iou": best,
                                "true_cls": truth[j][0]})
        for j, (c, b) in enumerate(truth):
            if claimed_by[j] == "correct":
                kind = "found"
            elif claimed_by[j] == "wrong class":
                kind = "found, wrong class"
            elif near[j]:
                kind = "poorly localised"
            else:
                kind = "missed"
            obj_records.append({"image": n, "kind": kind, "cls": c, "box": b})
    return det_records, obj_records


det_records, obj_records = analyse(test_out, test_boxes, test_labels)
det_kinds = ["correct", "wrong class", "duplicate", "poor box", "background"]
obj_kinds = ["found", "found, wrong class", "poorly localised", "missed"]

print(f"{len(det_records)} detections on {len(test_images)} test images:")
for kind in det_kinds:
    count = sum(r["kind"] == kind for r in det_records)
    print(f"   {kind:12} {count:5}   ({count / len(det_records):6.1%})")
print(f"\n{len(obj_records)} true objects:")
for kind in obj_kinds:
    count = sum(r["kind"] == kind for r in obj_records)
    print(f"   {kind:19} {count:5}   ({count / len(obj_records):6.1%})")

precision = sum(r["kind"] == "correct" for r in det_records) / len(det_records)
recall = sum(r["kind"] == "found" for r in obj_records) / len(obj_records)
matched = [r["iou"] for r in det_records if r["kind"] in ("correct", "wrong class")]
print(f"\nprecision {precision:.3f}   recall {recall:.3f}   "
      f"mean IoU of matched detections {np.mean(matched):.3f}")

### Looking at the errors

Numbers say how often; pictures say why. One row per category, up to six test images
each: true boxes in green, detections in red (dashed) with the predicted class at their
lower-right corner.

In [ ]:
def show_category(records: list, kind: str, n: int = 6) -> None:
    images = []
    for r in records:
        if r["kind"] == kind and r["image"] not in images:
            images.append(r["image"])
        if len(images) == n:
            break
    if not images:
        print(f"no '{kind}' at all")
        return
    fig, axes = plt.subplots(1, n, figsize=(2.2 * n, 2.5), squeeze=False)
    for ax in axes.flat:
        ax.axis("off")
    for ax, k in zip(axes.flat, images):
        draw_result(ax, k, decode(test_out[k]), title=f"{kind}: image {k}")
        ax.set_xticks([]); ax.set_yticks([])
    plt.tight_layout()
    plt.show()


for kind in ("wrong class", "poor box", "background", "duplicate"):
    show_category(det_records, kind)
show_category(obj_records, "missed")

### Which digits get confused, and how good are the boxes

For the detections that found their object (IoU ≥ 0.5), the class channels can be judged
exactly like the classifier of Exercise 09: a confusion matrix of true digit (row) against
predicted digit (column). And for the boxes, the distribution of IoU between each such
detection and its object: a box that is "correct" at IoU 0.5 can still be loose.

In [ ]:
m = np.zeros((10, 10), dtype=int)
for r in det_records:
    if r["kind"] in ("correct", "wrong class"):
        m[r["true_cls"], r["det"]["cls"]] += 1
print("true \\ predicted " + "".join(f"{d:>5}" for d in range(10)))
for d in range(10):
    print(f"{d:>16} " + "".join(f"{v:>5}" for v in m[d]))
print(f"\nclass accuracy on localised objects: {np.trace(m) / m.sum():.3f}")

fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 3.2))
a1.hist(matched, bins=np.linspace(0.5, 1.0, 26), color="tab:blue")
a1.set_xlabel("IoU with its object"); a1.set_ylabel("detections")
a1.set_title("box quality of the detections that found an object", fontsize=10)

found = np.zeros(10); total = np.zeros(10)
for r in obj_records:
    total[r["cls"]] += 1
    found[r["cls"]] += r["kind"] == "found"
a2.bar(range(10), found / total, color="tab:green")
a2.set_xticks(range(10)); a2.set_ylim(0, 1)
a2.set_xlabel("digit"); a2.set_ylabel("recall")
a2.set_title("fraction of objects found, per digit", fontsize=10)
plt.tight_layout()
plt.show()

### The threshold trades one error for another

Channel 0 is compared with a threshold $\tau$. Lower it and the detector says "object"
more often: fewer objects are missed, more detections fire on background. Raise it and the
opposite happens. Nothing is retrained; only the decoding changes.

In [ ]:
print(f"{'tau':>5}{'detections':>12}{'background':>12}{'missed':>8}{'precision':>11}{'recall':>8}")
for tau in (0.1, 0.3, 0.5, 0.7, 0.9):
    d_rec, o_rec = analyse(test_out, test_boxes, test_labels, threshold=tau)
    n_bg = sum(r["kind"] == "background" for r in d_rec)
    n_miss = sum(r["kind"] == "missed" for r in o_rec)
    prec = sum(r["kind"] == "correct" for r in d_rec) / max(len(d_rec), 1)
    rec = sum(r["kind"] == "found" for r in o_rec) / len(o_rec)
    print(f"{tau:>5.1f}{len(d_rec):>12}{n_bg:>12}{n_miss:>8}{prec:>11.3f}{rec:>8.3f}")

### What the errors say

Read the tables above channel by channel, since each kind of error points at one part of
the 15 numbers:

- **Objectness (channel 0) is nearly solved.** No detection fires on empty background, and
  after non-maximum suppression there are no duplicates; about 1 % of the objects are
  missed outright. The objectness map shows why NMS still matters: a digit whose centre
  lies near a cell border makes both neighbours fire (0.78 and 0.93 in the figure of
  Section 10), and only the higher score survives.
- **Class (channels 5–14) is the largest source of error.** About 5–6 % of the objects are
  found in the right place with the wrong digit — the confusions of Exercise 09 (4 ↔ 9,
  7 ↔ 9, 3 ↔ 5), now made by a cell that sees its digit through a $38 \times 38$ window
  shared with the neighbours. Exercise 09's classifier, trained on centred digits, did
  better (about 97 %): here the same network must also find the digit, with the same
  small budget of filters.
- **Box (channels 1–4) fails mostly on thin digits.** About four in five "poor box" detections are `1`s,
  whose boxes are 3–6 pixels wide: an error of a single pixel in $x$ removes a third of
  the intersection, and the IoU falls under 0.5 while the box is, to the eye, on the
  digit. IoU judges small objects harshly — a known property of the measure, not only of
  this network. The mean IoU of the matched detections, about 0.79, says the typical
  box is good but not pixel-exact.
- **The threshold barely matters here.** Between $\tau = 0.1$ and $0.7$, recall and
  precision move by about one point, because channel 0 is confident: almost every cell is
  near 0 or near 1. At $\tau = 0.9$ the detector starts to drop real digits.

Because the output separates *whether*, *where* and *what*, a detector's errors can be
diagnosed one channel group at a time — the reason this analysis follows Hoiem et al. and
does not stop at a single score.

## What to carry forward

- **A detector is a classifier per cell.** The variable-length set of objects becomes a
  fixed $8 \times 8 \times 15$ array by giving each object to the cell that holds its
  centre; the network predicts that array, and decoding turns it back into a set.
- **A $1 \times 1$ convolution is one dense layer shared by every cell**, so the head
  needs no knowledge of position: the same weights detect a digit anywhere.
- **The loss reads each part of the output with its own tool**: binary cross-entropy for
  "is there an object", squared error for "where", softmax cross-entropy for "which" —
  the last two only where an object exists.
- **IoU** measures box agreement; it decides duplicates in NMS and correctness in the
  evaluation, and it is harsh on small objects.

## Try it

1. Set `LAMBDA_NOOBJ = 1.0` and retrain. What happens to the objectness map, the number
   of missed objects and the precision?
2. Remove non-maximum suppression (`NMS_IOU = 1.01`). How many duplicates appear, and in
   which cells?
3. Let digits touch or overlap (`GAP = -6`). The encoder's check "two objects in one
   cell" will fire on some images: what would a detector need so that one cell can hold
   two objects? (YOLO's answer: several boxes per cell.)

### References

- J. Redmon, S. Divvala, R. Girshick, A. Farhadi. *You Only Look Once: Unified, Real-Time
  Object Detection.* CVPR 2016. [arXiv:1506.02640](https://arxiv.org/abs/1506.02640)
- D. Hoiem, Y. Chodpathumwan, Q. Dai. *Diagnosing Error in Object Detectors.* ECCV 2012.
- M. Everingham, L. Van Gool, C. K. I. Williams, J. Winn, A. Zisserman. *The PASCAL
  Visual Object Classes (VOC) Challenge.* IJCV 2010.